# LC 853 — Car Fleet
**Difficulty:** Medium &nbsp;|&nbsp; **Category:** Stack / Sorting
**Pattern:** Sort Descending, Stack of Arrival Times

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Sort cars by position
closest-to-target first. Compute each car's time to
reach the target. If a car's time is ≤ the fleet
ahead's time, it catches up and joins — same fleet.
Otherwise it forms a new fleet.
</div>

## Official Problem Statement

There are `n` cars going to the same destination
along a one-lane road. The destination is `target`
miles away.

You are given two integer array `position` and
`speed`, both of length `n`, where `position[i]`
is the position of the `i`th car and `speed[i]`
is the speed of the `i`th car (in miles per hour).

A car can never pass another car ahead of it, but
it can catch up to it and drive bumper to bumper at
the same speed. The faster car will slow down to
match the slower car's speed.

A car fleet is some non-empty set of cars driving
at the same position and same speed. Note that a
single car is also a car fleet.

If a car catches up to a car fleet right at the
destination point, it will still be considered as
one car fleet.

Return the number of car fleets that will arrive
at the destination.

**Example 1:**
```
Input:  target=12, position=[10,8,0,5,3], speed=[2,4,1,1,3]
Output: 3
```
**Example 2:**
```
Input:  target=10, position=[3], speed=[3]
Output: 1
```

**Constraints:**
- `n == position.length == speed.length`
- `1 <= n <= 10^5`
- `0 < target <= 10^6`
- `0 <= position[i] < target`
- `0 < speed[i] <= 10^6`
- All positions are unique

## What This Is Actually Asking

Cars drive toward a finish line on a single-lane
road — no passing allowed.
A faster car that catches a slower car in front
must slow down and travel together as one group.
Count how many distinct groups (fleets) arrive
at the finish line.

## Walk Through an Example by Hand

```
target=12
position=[10, 8, 0, 5, 3]
speed=   [ 2, 4, 1, 1, 3]

Pair and sort by position DESCENDING (closest first):
  (10,2) (8,4) (5,1) (3,3) (0,1)

Time to target = (target - pos) / speed:
  (10,2) -> (12-10)/2 = 1.0
  (8, 4) -> (12-8)/4  = 1.0
  (5, 1) -> (12-5)/1  = 7.0
  (3, 3) -> (12-3)/3  = 3.0
  (0, 1) -> (12-0)/1  = 12.0

Stack (times of distinct fleets):
  car(10,2): time=1.0  stack empty -> push  [1.0]
  car(8,4):  time=1.0  1.0 <= stack top 1.0
                       -> catches up -> NO push  [1.0]
  car(5,1):  time=7.0  7.0 > stack top 1.0
                       -> new fleet -> push  [1.0, 7.0]
  car(3,3):  time=3.0  3.0 <= stack top 7.0
                       -> catches up -> NO push  [1.0, 7.0]
  car(0,1):  time=12.0 12.0 > stack top 7.0
                       -> new fleet -> push  [1.0, 7.0, 12.0]

Answer: len(stack) = 3
```

## The Picture

```
Target = 12   (finish line)

Road:  0----3--5---8-10------12
            C  D   B  A      FINISH

Times to finish:
  A(10): (12-10)/2  = 1.0 hr
  B(8):  (12-8)/4   = 1.0 hr -> ties A -> same fleet
  D(5):  (12-5)/1   = 7.0 hr -> slower than A+B fleet
  C(3):  (12-3)/3   = 3.0 hr -> faster than D -> joins D
  E(0):  (12-0)/1   = 12.0 hr -> slowest of all

Fleet 1: A + B  (both arrive at t=1.0)
Fleet 2: D + C  (C catches D at t=7.0)
Fleet 3: E      (solo, arrives t=12.0)

Rule: if time_i <= time of fleet ahead -> join fleet
      if time_i >  time of fleet ahead -> new fleet
```

## When To Use This Pattern

- When cars (or jobs) can catch up to those ahead
  but not pass, think **sort descending by position,
  compare arrival times**
- When merging into a group depends on comparing
  with the most recent unmerged group, think **stack**
- When the answer is "count distinct groups",
  think **length of the stack at the end**
- When time = distance / speed, think **float
  division — no rounding needed for comparison**

## The Approach

Pair each car's position with its speed and sort
the pairs by position in descending order so the
car closest to the target is processed first.
For each car, compute its time to reach the target.
If that time exceeds the arrival time of the fleet
at the top of the stack, the car cannot catch up
and becomes a new fleet — push its time.
Otherwise it merges into the fleet ahead — skip.
Return the number of distinct times on the stack.

In [3]:
from typing import List  # type hints for the solution

In [4]:
def test_harness(func):
    tests = [
        # (target, position, speed, expected)
        (12, [10,8,0,5,3],  [2,4,1,1,3],  3),
        (10, [3],           [3],           1),
        (100,[0,2,4],       [4,2,1],       1),  # all merge
        (10, [6,8],         [3,2],         2),  # can't catch
        (10, [0,4,2],       [2,1,3],       1),  # all merge
        (10, [8,3,7,4,6],   [4,4,4,4,4],  5),  # all solo
        (10, [6,2],         [3,4],         2),  # faster behind
    ]

    passed = 0
    for i, (target, pos, spd, expected) in enumerate(tests):
        result = func(target, pos[:], spd[:])
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"Test {i+1}: {status} | "
            f"target={target} pos={pos} spd={spd} | "
            f"expected={expected} | got={result}"
        )

    print(f"\n{passed}/{len(tests)} tests passed")

In [52]:
from typing import List  # type hints for the solution
def carFleet(
    target: int,
    position: List[int],
    speed: List[int]
) -> int:
    """
    Return the number of car fleets reaching target.

    Sort cars by position descending. For each car,
    compute time = (target - pos) / speed. If time >
    stack top, the car is a new fleet (push). Otherwise
    it catches the fleet ahead (skip). Return len(stack).

    Time:  O(n log n) — dominated by sort
    Space: O(n) — stack holds at most n fleet times
    """
    # Sorted the pairs in reverse order in order of their poition
    # the farset the postion the closer to target, the more chance it becomes blocking
    # add time of the first element to the list|stack
    # add only items if they have time less than the one on top of stack
    #number of items on stack is our fleet length
    pairs = sorted(zip(position, speed), reverse=True)
    stack = []
    for p, s in pairs:
        t = (target -p )/s
        if  not stack or t > stack[-1]:
            stack.append(t)                  
    return len(stack)
'''
1
3
1
1
2
Test 1: PASSED | target=12 pos=[10, 8, 0, 5, 3] spd=[2, 4, 1, 1, 3] | expected=3 | got=3
Test 2: PASSED | target=10 pos=[3] spd=[3] | expected=1 | got=1
Test 3: PASSED | target=100 pos=[0, 2, 4] spd=[4, 2, 1] | expected=1 | got=1
Test 4: PASSED | target=10 pos=[6, 8] spd=[3, 2] | expected=2 | got=2
Test 5: PASSED | target=10 pos=[0, 4, 2] spd=[2, 1, 3] | expected=1 | got=1
Test 6: PASSED | target=10 pos=[8, 3, 7, 4, 6] spd=[4, 4, 4, 4, 4] | expected=5 | got=5
Test 7: PASSED | target=10 pos=[6, 2] spd=[3, 4] | expected=2 | got=2

7/7 tests passed

'''

print(carFleet(10,[0,4,2],       [2,1,3]))
# Quick debug — run this cell while building
print(carFleet(12, [10,8,0,5,3], [2,4,1,1,3]))  # 3
print(carFleet(10, [3],          [3]))            # 1
print(carFleet(100,[0,2,4],      [4,2,1]))        # 1
print(carFleet(10, [6,8],        [3,2]))          # 2
test_harness(carFleet)
pass
'''
def test_harness(func):
    tests = [
        # (target, position, speed, expected)
        (12, [10,8,0,5,3],  [2,4,1,1,3],  3),
        (10, [3],           [3],           1),
        (100,[0,2,4],       [4,2,1],       1),  # all merge
        (10, [6,8],         [3,2],         2),  # can't catch
        (10, [0,4,2],       [2,1,3],       1),  # all merge
        (10, [8,3,7,4,6],   [4,4,4,4,4],  5),  # all solo
        (10, [6,2],         [3,4],         2),  # faster behind
    ]

    passed = 0
    for i, (target, pos, spd, expected) in enumerate(tests):
        result = func(target, pos[:], spd[:])
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"Test {i+1}: {status} | "
            f"target={target} pos={pos} spd={spd} | "
            f"expected={expected} | got={result}"
        )

    print(f"\n{passed}/{len(tests)} tests passed")
'''


1
3
1
1
2
Test 1: PASSED | target=12 pos=[10, 8, 0, 5, 3] spd=[2, 4, 1, 1, 3] | expected=3 | got=3
Test 2: PASSED | target=10 pos=[3] spd=[3] | expected=1 | got=1
Test 3: PASSED | target=100 pos=[0, 2, 4] spd=[4, 2, 1] | expected=1 | got=1
Test 4: PASSED | target=10 pos=[6, 8] spd=[3, 2] | expected=2 | got=2
Test 5: PASSED | target=10 pos=[0, 4, 2] spd=[2, 1, 3] | expected=1 | got=1
Test 6: PASSED | target=10 pos=[8, 3, 7, 4, 6] spd=[4, 4, 4, 4, 4] | expected=5 | got=5
Test 7: PASSED | target=10 pos=[6, 2] spd=[3, 4] | expected=2 | got=2

7/7 tests passed


In [ ]:
# Uncomment and run when solution is ready
# test_harness(carFleet)

## Complexity

| Approach | Time | Space |
|---|---|---|
| Brute force — simulate each step | O(n²) | O(n) |
| Sort + stack of arrival times | O(n log n) | O(n) |

The sort dominates — the stack pass is a single
O(n) scan after sorting.

## Real World Connection

At Citi, batch ETL jobs compete for a shared
processing cluster on a single-lane queue — faster
jobs queued behind a slow one must wait and
effectively form a "fleet" bottlenecked by the
slowest member.
The Car Fleet pattern counts how many distinct
throughput groups actually emerge after queuing,
which directly informs the SLA analysis: how many
independent completion windows should the on-call
team expect?
On AWS, Step Functions state-machine executions
that fan out to the same Lambda concurrency limit
form fleets in the same way — sort by start time,
compare projected finish times, count groups.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra